# 07. Interaction 피처 + missing_count + Native NaN 실험

팀원의 XGBoost 리포트(Balanced Accuracy 0.95024)를 참고해서, 저희 04 baseline(0.94987)에 다음 3가지를 각각/합쳐서 테스트합니다.

1. **Interaction 피처**: 라벨 생성 규칙(`sleep_duration` × `stress_level` × `physical_activity_level`)을 명시적 binary/categorical 피처로 주입
2. **missing_count**: 핵심 3피처 결측 개수(0~3), 전체 13피처 결측 개수 — 저희가 찾은 병목(결측 2개 이상 겹친 행)을 모델이 더 쉽게 구분하도록
3. **Native NaN 처리**: 나머지 수치형(heart_rate, bmi, calorie_expenditure, step_count, exercise_duration, water_intake)을 median으로 채우지 않고 NaN 그대로 둬서 LightGBM의 네이티브 결측 분기 학습에 맡김

04와 동일한 5-fold, 동일한 튜닝된 하이퍼파라미터, 동일한 사전확률 보정 결정규칙을 사용해서 **Feature Set만 바꿔가며 비교**합니다 (03/05/06과 같은 방법론).

커널: **Python (teammate)**

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics import balanced_accuracy_score, accuracy_score
import lightgbm as lgb

SEED = 42
N_FOLDS = 5
DATA_DIR = Path("../playground-series-s6e7")
OUT_DIR = DATA_DIR / "processed"
TARGET = "health_condition"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
folds = pd.read_csv(OUT_DIR / "cv_folds.csv")
train = train.merge(folds, on="id", how="left")
assert train["fold"].isna().sum() == 0

ORIGINAL_FEATURE_COLS = [
    "sleep_duration", "heart_rate", "bmi", "calorie_expenditure", "step_count",
    "exercise_duration", "water_intake", "diet_type", "stress_level",
    "sleep_quality", "physical_activity_level", "smoking_alcohol", "gender",
]
print(train.shape, test.shape)

(690088, 16) (295753, 14)


## 1. 결측 복구 피처 (04와 동일)

In [2]:
def add_recovery_features(df, step_tertiles, sleep_quality_cond):
    df = df.copy()
    activity_proxy = pd.cut(
        df["step_count"], bins=[-np.inf, step_tertiles[0], step_tertiles[1], np.inf],
        labels=["sedentary", "moderate", "active"],
    ).astype(object)
    df["physical_activity_level_recovered"] = df["physical_activity_level"]
    missing_activity = df["physical_activity_level"].isna()
    df.loc[missing_activity, "physical_activity_level_recovered"] = activity_proxy[missing_activity]
    df["physical_activity_level_recovered"] = df["physical_activity_level_recovered"].fillna("moderate")

    df["sleep_duration_recovered"] = df["sleep_duration"]
    missing_sleep = df["sleep_duration"].isna()
    fallback_median = sleep_quality_cond.get("missing", sleep_quality_cond["average"])
    mapped = df.loc[missing_sleep, "sleep_quality"].map(sleep_quality_cond).fillna(fallback_median)
    df.loc[missing_sleep, "sleep_duration_recovered"] = mapped

    df["stress_level_isnull"] = df["stress_level"].isna().astype(np.int8)
    df["sleep_duration_isnull"] = df["sleep_duration"].isna().astype(np.int8)
    df["physical_activity_level_isnull"] = df["physical_activity_level"].isna().astype(np.int8)
    return df


step_tertiles = train["step_count"].quantile([1/3, 2/3]).values
sleep_quality_cond = train.groupby("sleep_quality")["sleep_duration"].median().to_dict()
sleep_quality_cond["missing"] = train["sleep_duration"].median()

train = add_recovery_features(train, step_tertiles, sleep_quality_cond)
test = add_recovery_features(test, step_tertiles, sleep_quality_cond)

## 2. 아이디어 1 — Interaction 피처 (라벨 규칙 명시적 주입)

In [3]:
def add_interaction_features(df):
    df = df.copy()
    sd = df["sleep_duration_recovered"]
    stress = df["stress_level"]
    activity = df["physical_activity_level_recovered"]

    df["sleep_lt6"] = (sd < 6).astype(np.int8)
    df["sleep_7plus"] = (sd >= 7).astype(np.int8)
    df["stress_high"] = (stress == "high").astype(np.int8)
    df["stress_low"] = (stress == "low").astype(np.int8)
    df["activity_active"] = (activity == "active").astype(np.int8)
    df["activity_sedentary"] = (activity == "sedentary").astype(np.int8)

    df["high_stress_short_sleep"] = (df["stress_high"] & df["sleep_lt6"]).astype(np.int8)
    df["low_stress_good_sleep"] = (df["stress_low"] & df["sleep_7plus"]).astype(np.int8)
    df["low_stress_active"] = (df["stress_low"] & df["activity_active"]).astype(np.int8)
    df["low_stress_good_sleep_active"] = (
        df["stress_low"] & df["sleep_7plus"] & df["activity_active"]
    ).astype(np.int8)

    # 라벨 생성 규칙을 통째로 하나의 카테고리 피처로 주입 (A/B/C/D + stress 결측 별도 처리)
    def branch(row):
        if pd.isna(row["stress_level"]) or row["stress_level"] == "missing":
            return "stress_missing"
        if row["sleep_duration_recovered"] < 6:
            return "A_unhealthy" if row["stress_level"] == "high" else "B_atrisk"
        if row["sleep_duration_recovered"] >= 7:
            if row["stress_level"] == "low" and row["physical_activity_level_recovered"] == "active":
                return "C_fit"
            return "D_atrisk"
        return "D_atrisk_mid"

    df["rule_branch"] = df.apply(branch, axis=1)
    return df


train = add_interaction_features(train)
test = add_interaction_features(test)

INTERACTION_BINARY_COLS = [
    "sleep_lt6", "sleep_7plus", "stress_high", "stress_low",
    "activity_active", "activity_sedentary", "high_stress_short_sleep",
    "low_stress_good_sleep", "low_stress_active", "low_stress_good_sleep_active",
]

train["rule_branch"].value_counts(normalize=True).round(4)

rule_branch
D_atrisk          0.3731
D_atrisk_mid      0.2892
stress_missing    0.1200
B_atrisk          0.1080
A_unhealthy       0.0637
C_fit             0.0459
Name: proportion, dtype: float64

## 3. 아이디어 2 — missing_count 집계 피처

In [4]:
CORE_MISSING_COLS = ["stress_level_isnull", "sleep_duration_isnull", "physical_activity_level_isnull"]

for df in (train, test):
    df["core_missing_count"] = df[CORE_MISSING_COLS].sum(axis=1).astype(np.int8)
    df["all_missing_count"] = df[ORIGINAL_FEATURE_COLS].isna().sum(axis=1).astype(np.int8)

MISSING_COUNT_COLS = ["core_missing_count", "all_missing_count"]
train[MISSING_COUNT_COLS].describe()

,core_missing_count,all_missing_count
count,690088.000000,690088.000000
mean,0.283197,0.651360
std,0.503750,0.772452
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,1.000000,1.000000
max,3.000000,6.000000


## 4. 나머지 전처리 — 두 버전(median 채움 vs NaN 유지) 모두 준비

아이디어 3(Native NaN)을 위해, "나머지 수치형" 컬럼을 median으로 채운 버전과 NaN 그대로 둔 버전을 둘 다 만들어둡니다.

In [5]:
OTHER_NUMERIC_COLS = ["heart_rate", "bmi", "calorie_expenditure", "step_count", "exercise_duration", "water_intake"]
ORDINAL_COLS = {
    "stress_level": ["low", "medium", "high"],
    "sleep_quality": ["poor", "average", "good"],
    "physical_activity_level_recovered": ["sedentary", "moderate", "active"],
    "smoking_alcohol": ["no", "occasional", "yes"],
}
NOMINAL_COLS = ["diet_type", "gender", "rule_branch"]
FLAG_COLS = ["stress_level_isnull", "sleep_duration_isnull", "physical_activity_level_isnull"]

# median 채움 버전 컬럼 (기존 04 방식)
other_numeric_medians = train[OTHER_NUMERIC_COLS].median()
for df in (train, test):
    for col in OTHER_NUMERIC_COLS:
        df[f"{col}_filled"] = df[col].fillna(other_numeric_medians[col])
        # 원본 col은 NaN 유지한 채로 그대로 둠 (native NaN 버전에서 사용)

categorical_cols = list(ORDINAL_COLS.keys()) + ["diet_type", "gender"]
for df in (train, test):
    for col in categorical_cols:
        df[col] = df[col].fillna("missing")

for col, order in ORDINAL_COLS.items():
    categories = order + ["missing"]
    encoder = OrdinalEncoder(categories=[categories])
    train[col] = encoder.fit_transform(train[[col]])
    test[col] = encoder.transform(test[[col]])

train_ohe = pd.get_dummies(train[NOMINAL_COLS], prefix=NOMINAL_COLS)
test_ohe = pd.get_dummies(test[NOMINAL_COLS], prefix=NOMINAL_COLS)
test_ohe = test_ohe.reindex(columns=train_ohe.columns, fill_value=0)
train = pd.concat([train.drop(columns=NOMINAL_COLS), train_ohe], axis=1)
test = pd.concat([test.drop(columns=NOMINAL_COLS), test_ohe], axis=1)

target_encoder = LabelEncoder()
train["target_enc"] = target_encoder.fit_transform(train[TARGET])
lgb_class_order = list(target_encoder.classes_)
train_priors = train[TARGET].value_counts(normalize=True).to_dict()

print("준비 완료")

준비 완료


## 5. Feature Set 5종 정의

04와 동일한 base(순서형 4 + 결측플래그 3 + one-hot(diet_type, gender))에 각 아이디어를 하나씩, 그리고 다 합쳐서 추가.

In [6]:
BASE_ORDINAL = list(ORDINAL_COLS.keys())
BASE_ONEHOT = [c for c in train_ohe.columns if not c.startswith("rule_branch")]
RULE_BRANCH_ONEHOT = [c for c in train_ohe.columns if c.startswith("rule_branch")]

MEDIAN_FILLED_NUMERIC = [f"{c}_filled" for c in OTHER_NUMERIC_COLS]

FEATURE_SETS = {
    "V0_baseline (04)": (
        ["sleep_duration_recovered"] + MEDIAN_FILLED_NUMERIC + BASE_ORDINAL + FLAG_COLS + BASE_ONEHOT
    ),
    "V1_+interaction": (
        ["sleep_duration_recovered"] + MEDIAN_FILLED_NUMERIC + BASE_ORDINAL + FLAG_COLS + BASE_ONEHOT
        + INTERACTION_BINARY_COLS + RULE_BRANCH_ONEHOT
    ),
    "V2_+missing_count": (
        ["sleep_duration_recovered"] + MEDIAN_FILLED_NUMERIC + BASE_ORDINAL + FLAG_COLS + BASE_ONEHOT
        + MISSING_COUNT_COLS
    ),
    "V3_+native_nan": (
        ["sleep_duration_recovered"] + OTHER_NUMERIC_COLS + BASE_ORDINAL + FLAG_COLS + BASE_ONEHOT
    ),
    "V4_+all_combined": (
        ["sleep_duration_recovered"] + OTHER_NUMERIC_COLS + BASE_ORDINAL + FLAG_COLS + BASE_ONEHOT
        + INTERACTION_BINARY_COLS + RULE_BRANCH_ONEHOT + MISSING_COUNT_COLS
    ),
}

for name, cols in FEATURE_SETS.items():
    print(name, "->", len(cols), "features")

V0_baseline (04) -> 22 features
V1_+interaction -> 38 features
V2_+missing_count -> 24 features
V3_+native_nan -> 22 features
V4_+all_combined -> 40 features


## 6. 5개 Feature Set 각각 5-fold CV (04의 튜닝된 파라미터 그대로 사용)

In [7]:
def prior_corrected_predict(proba, class_order, priors):
    prior_arr = np.array([priors[c] for c in class_order])
    scores = proba / prior_arr
    return np.array(class_order)[scores.argmax(axis=1)]


tuned_params = dict(
    objective="multiclass", num_class=3, random_state=SEED, verbosity=-1,
    n_estimators=500,
    learning_rate=0.047792122422826176,
    num_leaves=19,
    max_depth=9,
    min_child_samples=172,
    subsample=0.8929201812783885,
    colsample_bytree=0.7318659835628288,
    reg_alpha=0.0005023614837892232,
    reg_lambda=0.32275087452118445,
)

results = []
oof_store = {}
for name, feature_cols in FEATURE_SETS.items():
    oof_proba = np.zeros((len(train), 3))
    for fold in range(N_FOLDS):
        tr_idx = train["fold"] != fold
        va_idx = train["fold"] == fold
        X_tr, y_tr = train.loc[tr_idx, feature_cols], train.loc[tr_idx, "target_enc"]
        X_va, y_va = train.loc[va_idx, feature_cols], train.loc[va_idx, "target_enc"]

        model = lgb.LGBMClassifier(**tuned_params)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
        oof_proba[va_idx.values] = model.predict_proba(X_va)

    pred = prior_corrected_predict(oof_proba, lgb_class_order, train_priors)
    ba = balanced_accuracy_score(train[TARGET].values, pred)
    acc = accuracy_score(train[TARGET].values, pred)
    results.append({"feature_set": name, "n_features": len(feature_cols), "accuracy": acc, "balanced_accuracy": ba})
    oof_store[name] = oof_proba
    print(f"{name}: BA={ba:.5f} (n_features={len(feature_cols)})")

results_df = pd.DataFrame(results)
results_df["vs_04_baseline"] = results_df["balanced_accuracy"] - 0.94987
results_df

V0_baseline (04): BA=0.94988 (n_features=22)


V1_+interaction: BA=0.94964 (n_features=38)


V2_+missing_count: BA=0.94977 (n_features=24)


V3_+native_nan: BA=0.94977 (n_features=22)


V4_+all_combined: BA=0.94970 (n_features=40)


,feature_set,n_features,accuracy,balanced_accuracy,vs_04_baseline
0,V0_baseline (04),22,0.939157,0.949880,0.000010
1,V1_+interaction,38,0.939590,0.949639,-0.000231
2,V2_+missing_count,24,0.939032,0.949766,-0.000104
3,V3_+native_nan,22,0.939124,0.949770,-0.000100
4,V4_+all_combined,40,0.939274,0.949701,-0.000169


## 7. 최고 성능 Feature Set으로 최종 제출 파일 생성 (04 대비 개선된 경우만)

In [8]:
best_row = results_df.sort_values("balanced_accuracy", ascending=False).iloc[0]
print("최고 성능 Feature Set:", best_row["feature_set"], "BA:", best_row["balanced_accuracy"])

if best_row["balanced_accuracy"] > 0.94987:
    best_feature_cols = FEATURE_SETS[best_row["feature_set"]]
    final_model = lgb.LGBMClassifier(**tuned_params)
    final_model.fit(train[best_feature_cols], train["target_enc"])
    test_proba = final_model.predict_proba(test[best_feature_cols])
    test_pred = prior_corrected_predict(test_proba, lgb_class_order, train_priors)
    submission = pd.DataFrame({"id": test["id"], TARGET: test_pred})
    submission.to_csv(OUT_DIR / "submission_v5_interaction.csv", index=False)
    print("개선 확인, 저장:", OUT_DIR / "submission_v5_interaction.csv")
    print(submission[TARGET].value_counts(normalize=True))

    gain_model = lgb.LGBMClassifier(**tuned_params, importance_type="gain")
    gain_model.fit(train[best_feature_cols], train["target_enc"])
    importance = pd.Series(gain_model.feature_importances_, index=best_feature_cols).sort_values(ascending=False)
    print("\nTop 15 feature importance (gain):")
    print(importance.head(15))
else:
    print("04 대비 개선 없음 -> submission_v2_tuned.csv를 그대로 유지")

최고 성능 Feature Set: V0_baseline (04) BA: 0.9498795129436178


개선 확인, 저장: ../playground-series-s6e7/processed/submission_v5_interaction.csv
health_condition
at-risk      0.810061
unhealthy    0.115884
fit          0.074055
Name: proportion, dtype: float64



Top 15 feature importance (gain):
stress_level                         2.580963e+06
sleep_duration_recovered             1.944909e+06
physical_activity_level_recovered    9.014529e+05
bmi_filled                           2.581051e+05
sleep_duration_isnull                1.574085e+05
exercise_duration_filled             8.988002e+04
step_count_filled                    7.573225e+04
sleep_quality                        7.255541e+04
smoking_alcohol                      3.442852e+04
physical_activity_level_isnull       3.432868e+04
water_intake_filled                  2.176707e+04
heart_rate_filled                    1.449154e+04
calorie_expenditure_filled           1.400773e+04
stress_level_isnull                  4.522446e+03
diet_type_non-veg                    1.527842e+03
dtype: float64
